# 📊 Evaluation Deep Dive — The 6-Pillar Faithfulness Framework

CircuitKit evaluates discovered circuits using 6 complementary pillars. Each answers a different question about whether the circuit truly explains the model's behavior.

| # | Pillar | Question | Cost |
|---|--------|----------|------|
| 1 | Causal Patching | Does ablating the circuit break the behavior? | Fast |
| 2 | Ablation | Is the circuit sufficient to reproduce the behavior? | Fast |
| 3 | Stability | Is the circuit reproducible across random seeds? | Expensive |
| 4 | Robustness | Is the circuit stable under different corruptions? | Moderate |
| 5 | Baselines | Does the circuit beat random/magnitude baselines? | Moderate |
| 6 | Generalization | Does the circuit transfer to related tasks? | Expensive |

- **Model:** `meta-llama/Llama-3.2-1B` (~1B params)
- **Task:** IOI (with `double_io` as generalization target)
- **Runtime:** ~25-35 min on Colab T4

---

## Setup

In [ ]:
# ⚠️ Must be set before any CUDA context is created.
# If you've already run GPU cells in a prior notebook, restart the runtime first.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

import warnings
warnings.filterwarnings('ignore', module='transformer_lens')
warnings.filterwarnings('ignore', module='lm_eval')
warnings.filterwarnings('ignore', message='.*pretrained.*model kwarg.*')

import gc
import json
import torch
from circuitkit import Pipeline

print(f"CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

## Step 1: Discover a Circuit

We need a circuit before we can evaluate it.

In [ ]:
pipe = Pipeline(
    "meta-llama/Llama-3.2-1B",
    task="ioi",
    output_dir="./results/eval_deep_dive",
)

pipe.discover(
    algorithm="eap-ig",
    level="node",
    sparsity=0.3,
    n_examples=256,
    batch_size=2,
    ig_steps=5,
)

print(pipe._circuit)
print(f"Artifact: {pipe._artifact_path}")

## Section A: Pillars 1 & 2 — Patching and Ablation (Fast)

These are the cheapest evaluations and should always be run first.

- **Patching**: Ablate (zero out) the circuit nodes → measure how much the target behavior degrades. A faithful circuit should show a large performance drop.
- **Ablation**: Keep only the circuit nodes, ablate everything else → measure retained performance. A sufficient circuit should retain most of the behavior.

In [ ]:
pipe.evaluate(
    pillars=["patching", "ablation"],
    n_examples=256,
)

if pipe._eval_report:
    print(json.dumps(pipe._eval_report, indent=2, default=str))

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section B: Pillar 5 — Baseline Comparison

Does the circuit beat random selection? Pillar 5 compares the discovered circuit's scores against random and magnitude-based baselines at the same sparsity level.

In [ ]:
pipe.evaluate(
    pillars=["baselines"],
    n_examples=256,
)

if pipe._eval_report:
    print(json.dumps(pipe._eval_report, indent=2, default=str))

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section C: Pillar 4 — Robustness

How stable is the circuit under different corruption strategies? Pillar 4 varies the corruption method and checks whether the same nodes remain important.

In [ ]:
pipe.evaluate(
    pillars=["robustness"],
    n_examples=256,
)

if pipe._eval_report:
    print(json.dumps(pipe._eval_report, indent=2, default=str))

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section D: Pillar 3 — Stability

Run discovery multiple times with different random seeds and measure circuit overlap (Jaccard/Dice similarity). A stable circuit should be reproducible.

> This is expensive — each stability run re-runs discovery from scratch.

In [ ]:
pipe.evaluate(
    pillars=["stability"],
    n_examples=64,  # lower than 256 -- each stability run re-runs discovery
    n_stability_runs=3,  # keep low for demo; use 5-10 for research
)

if pipe._eval_report:
    print(json.dumps(pipe._eval_report, indent=2, default=str))

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section E: Pillar 6 — Generalization

Does the circuit transfer to a related but different task? For IOI, the standard generalization target is `double_io` (a variant with two indirect objects).

In [ ]:
pipe.evaluate(
    pillars=["generalization"],
    n_examples=64,
    target_task="double_io",
)

if pipe._eval_report:
    print(json.dumps(pipe._eval_report, indent=2, default=str))

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section F: All 6 Pillars at Once

For convenience, you can run all pillars in a single call. The framework runs them in optimal cost order internally.

> **Note:** If you already ran Sections A–E above, this cell re-runs discovery and all evaluations on a fresh pipeline. You can skip it to save time — it's included to demonstrate the single-call API.

In [ ]:
# Run a fresh discovery to get a clean evaluation
pipe_full = Pipeline(
    "meta-llama/Llama-3.2-1B",
    task="ioi",
    output_dir="./results/eval_full",
)
pipe_full.discover(algorithm="eap-ig", n_examples=256, batch_size=2, ig_steps=5)

# All pillars
pipe_full.evaluate(
    pillars="all",
    n_examples=64,
    n_stability_runs=3,
    target_task="double_io",
)

if pipe_full._eval_report:
    print(json.dumps(pipe_full._eval_report, indent=2, default=str))

## How to Read Faithfulness Numbers

A circuit is "good enough" when:

- **Patching score** is high (ablating circuit → large performance drop)
- **Ablation score** is high (circuit alone retains most behavior)
- **Stability** Jaccard > 0.5 across seeds
- **Baselines** — circuit outperforms random selection at the same sparsity
- **Robustness** — scores are stable across corruption types
- **Generalization** — some transfer to related tasks (depends on task similarity)

No single pillar is definitive. The framework is designed to give a **multi-dimensional** view of circuit quality.